<a href="https://colab.research.google.com/github/smagadi/AIML/blob/master/GenAI/chatGPTPowerdTechanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [69]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
!pip install phidata openai yfinance googlesearch-python
!pip install pycountry
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime
from google.colab import userdata
import openai
import os

In [70]:
key =userdata.get('OPENAI')

# Retrieve the API key from Colab Secrets
# Try setting the API key as an environment variable:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI')


In [71]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime

def plot_candlestick_with_indicators(ticker, start_date='2020-01-01', end_date=datetime.today().strftime('%Y-%m-%d')):
    """
    Creates a candlestick chart with SMA, EMA, Bollinger Bands, and VWAP indicators.

    Parameters:
    ticker (str): Stock symbol (e.g., 'AAPL')
    start_date (str): Start date in 'YYYY-MM-DD' format
    end_date (str): End date in 'YYYY-MM-DD' format
    """

    # Fetch stock data
    stock_data = yf.Ticker(ticker).history(start=start_date, end=end_date)

    # Check if data exists
    if stock_data.empty:
        raise ValueError(f"No data found for {ticker} between {start_date} and {end_date}")

    # Calculate 20-day SMA
    stock_data['SMA_20'] = stock_data['Close'].rolling(window=20).mean()

    # Calculate 20-day EMA
    stock_data['EMA_20'] = stock_data['Close'].ewm(span=20, adjust=False).mean()

    # Calculate Bollinger Bands
    stock_data['BB_Mid'] = stock_data['Close'].rolling(window=20).mean()
    stock_data['BB_Upper'] = stock_data['BB_Mid'] + (2 * stock_data['Close'].rolling(window=20).std())
    stock_data['BB_Lower'] = stock_data['BB_Mid'] - (2 * stock_data['Close'].rolling(window=20).std())

    # Calculate VWAP
    stock_data['VWAP'] = (stock_data['Volume'] * (stock_data['High'] + stock_data['Low'] + stock_data['Close']) / 3).cumsum() / stock_data['Volume'].cumsum()

    # Interpolate NaN values
    stock_data['SMA_20'] = stock_data['SMA_20'].interpolate(method='linear', limit_direction='both')
    stock_data['BB_Mid'] = stock_data['BB_Mid'].interpolate(method='linear', limit_direction='both')
    stock_data['BB_Upper'] = stock_data['BB_Upper'].interpolate(method='linear', limit_direction='both')
    stock_data['BB_Lower'] = stock_data['BB_Lower'].interpolate(method='linear', limit_direction='both')

    # Create candlestick chart
    fig = go.Figure(data=[go.Candlestick(
        x=stock_data.index,
        open=stock_data['Open'],
        high=stock_data['High'],
        low=stock_data['Low'],
        close=stock_data['Close'],
        name='Candlestick'
    )])

    # Add SMA line
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['SMA_20'], mode='lines', name='SMA 20', line=dict(color='blue', width=2)))

    # Add EMA line
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['EMA_20'], mode='lines', name='EMA 20', line=dict(color='orange', width=2)))

    # Add Bollinger Bands
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Upper'], mode='lines', name='Bollinger Upper', line=dict(color='green', dash='dash', width=1)))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Lower'], mode='lines', name='Bollinger Lower', line=dict(color='red', dash='dash', width=1)))
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['BB_Mid'], mode='lines', name='Bollinger Middle', line=dict(color='black', width=1)))

    # Add VWAP line
    fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['VWAP'], mode='lines', name='VWAP', line=dict(color='purple', width=2)))

    # Customize layout
    fig.update_layout(
        title=f'{ticker} Candlestick Chart with Indicators ({start_date} to {end_date})',
        xaxis_title='Date',
        yaxis_title='Price (USD)',
        xaxis_rangeslider_visible=False,
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    # Show plot
    fig.show()
    # Return stock data
    return stock_data

# Example usage
sd = plot_candlestick_with_indicators('AAPL')
print(sd)


                                 Open        High         Low       Close  \
Date                                                                        
2020-01-02 00:00:00-05:00   71.721019   72.776598   71.466812   72.716072   
2020-01-03 00:00:00-05:00   71.941336   72.771752   71.783969   72.009125   
2020-01-06 00:00:00-05:00   71.127851   72.621631   70.876060   72.582893   
2020-01-07 00:00:00-05:00   72.592594   72.849224   72.021231   72.241547   
2020-01-08 00:00:00-05:00   71.943759   73.706279   71.943759   73.403648   
...                               ...         ...         ...         ...   
2025-02-21 00:00:00-05:00  245.949997  248.690002  245.220001  245.550003   
2025-02-24 00:00:00-05:00  244.929993  248.860001  244.419998  247.100006   
2025-02-25 00:00:00-05:00  248.000000  250.000000  244.910004  247.039993   
2025-02-26 00:00:00-05:00  244.330002  244.979996  239.130005  240.360001   
2025-02-27 00:00:00-05:00  239.410004  242.460007  237.059998  237.300003   

In [78]:
# Example usage
sd= plot_candlestick_with_indicators('WMT', '2024-09-01')

In [79]:
def ask_gpt_for_analysis_gpt4o_mini(stock_ticker, recent_stock_data):
    """
    Asks GPT-4o Mini for a financial analysis of the given stock data.

    Parameters:
        - stock_ticker: The ticker symbol of the stock.
        - recent_stock_data: A DataFrame containing recent financial data.

    Returns:
        - GPT-4o Mini's response as a string.
    """

    # Prepare the prompt for GPT-4o Mini
    prompt = f"""
You are a financial analyst. Based on the following recent data for {stock_ticker}, provide:
1. A recommendation (Buy/Sell/Hold) with reasons.
2. A target price with reasons.

Here is the data:

{recent_stock_data.tail(50).to_string(index=False)}

Please analyze this data and provide your insights.
"""

    # Call the OpenAI API using GPT-4o Mini
    # Update: Use client.chat.completions.create instead of openai.ChatCompletion.create
    # and import OpenAI
    from openai import OpenAI
    client = OpenAI()

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Specify GPT-4o Mini as the model #gpt-4o-mini
        messages=[
            {"role": "system", "content": "You are a financial analyst providing actionable insights."},
            {"role": "user", "content": prompt}
        ]
    )

    # Extract and return GPT's response
    # Update: Access the content from the new response structure
    return response.choices[0].message.content

In [80]:
# Ask GPT-4o Mini for analysis based on recent data
gpt_response = ask_gpt_for_analysis_gpt4o_mini("WMT", sd)

print("\nGPT-4o Mini Analysis:")
print(gpt_response)


GPT-4o Mini Analysis:
Based on the data provided, here are my insights and recommendations:

1. Recommendation: **Hold**
   - The stock price has been fluctuating within a range without a clear upward or downward trend. It is trading close to its 20-day Simple Moving Average (SMA) and Exponential Moving Average (EMA), indicating a lack of clear direction in the short term.
   - The Bollinger Bands (BB) are not showing any extreme levels, suggesting a neutral stance. However, the stock has been trading closer to the lower band recently, which might indicate potential undervaluation.
   - The Volume Weighted Average Price (VWAP) has been relatively stable, indicating consistent trading patterns.

2. Target Price: $97.00
   - Considering the recent price fluctuations and the overall stability in trading patterns, I would set a conservative target price of $97.00.
   - This target price takes into account the current support levels indicated by the Bollinger Bands and the recent trading a